In [2]:
import pandas as pd
import pymysql
from sqlalchemy import inspect, text
from core.database_utils import get_db_engine, get_database_inventory

In [3]:
# Solo necesitas llamar a la función
engine = get_db_engine()

if engine:
    # Ahora puedes obtener el inventario completo automáticamente
    inventory = get_database_inventory()

--- Connection to 52.249.30.165 established successfully ---
--- Connection to 52.249.30.165 established successfully ---
--- Database Inventory: granerosguerra_copiaenrique ---
Total tables found: 0



In [4]:


def run_query(sql_query, description="Data"):
    """
    Executes a SQL query and returns a pandas DataFrame.
    :param sql_query: The SQL text or SQLAlchemy text() object.
    :param description: A label for printing status messages.
    :return: Pandas DataFrame or None if failed.
    """
    try:
        # 1. Ensure the query is wrapped in text() if it's a raw string
        if isinstance(sql_query, str):
            sql_query = text(sql_query)
            
        with engine.connect() as conn:
            df = pd.read_sql_query(sql_query, conn)
            
        # 2. Check and report status
        if df.empty:
            print(f"--- Warning: No records found for [{description}]. ---")
            return df
        else:
            print(f"--- Success: [{description}] retrieved with {len(df)} rows. ---")
            return df
            
    except Exception as e:
        print(f"--- Critical Error executing [{description}]: {e} ---")
        return None



In [ ]:

# Definir la consulta para listar tablas del esquema 'dbo' o todos
query_tables = text("""
    SELECT 
    s.name AS [Schema],
    t.name AS [Table],
    c.name AS [Column],
    pc.name AS [DataType],
    c.max_length AS [MaxLength],
    c.is_nullable AS [IsNullable]
FROM sys.columns c
JOIN sys.tables t ON c.object_id = t.object_id
JOIN sys.schemas s ON t.schema_id = s.schema_id
JOIN sys.types pc ON c.user_type_id = pc.user_type_id
WHERE t.name IN ('CATALOGO_CLIENTES_DATA', 'COBRANZA_DATA', 'CREDITO_DATA', 'DESTINOS_CLIENTES')
ORDER BY t.name, c.column_id;
""")



In [16]:
df_tab = run_query(query_tables, "Table inventory")
print(df_tab)


--- Success: [Table inventory] retrieved with 45 rows. ---
     Schema                   Table             Column  DataType  MaxLength  \
0   proadel  CATALOGO_CLIENTES_DATA             CODIGO  nvarchar         40   
1   proadel  CATALOGO_CLIENTES_DATA             NOMBRE  nvarchar        300   
2   proadel  CATALOGO_CLIENTES_DATA          DIRECCION  nvarchar        300   
3   proadel  CATALOGO_CLIENTES_DATA           TELEFONO  nvarchar        100   
4   proadel  CATALOGO_CLIENTES_DATA  LIMITE_DE_CREDITO   decimal          9   
5   proadel  CATALOGO_CLIENTES_DATA      TOTAL_CREDITO   decimal          9   
6   proadel  CATALOGO_CLIENTES_DATA     TOTAL_COBRANZA   decimal          9   
7   proadel  CATALOGO_CLIENTES_DATA    TOTAL_DEV_VENTA   decimal          9   
8   proadel  CATALOGO_CLIENTES_DATA      SALDO_INICIAL   decimal          9   
9   proadel  CATALOGO_CLIENTES_DATA             ESTADO  nvarchar        100   
10  proadel  CATALOGO_CLIENTES_DATA   TOTAL_POR_ABONAR   decimal        

In [ ]:

# =============================================================================
# SECTION 1: IMPORTS, CONFIGURATION & DATA INGESTION
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sqlalchemy import text

# Reference date for all recency/age calculations
ANALYSIS_DATE = pd.Timestamp.today().normalize()

# Global chart style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi"     : 110,
    "axes.titlesize" : 13,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

# -- 1.1  Customer catalog ------------------------------------------------
df_customers = run_query(
    "SELECT * FROM proadel.CATALOGO_CLIENTES_DATA",
    "Customer Catalog"
)

# -- 1.2  Payment / collection records ------------------------------------
df_payments = run_query(
    "SELECT * FROM proadel.COBRANZA_DATA",
    "Payment (Cobranza) Data"
)

# -- 1.3  Credit / sales records ------------------------------------------
df_credits = run_query(
    "SELECT * FROM proadel.CREDITO_DATA",
    "Credit (Ventas) Data"
)

# -- 1.4  Customer-destination mapping ------------------------------------
df_destinations = run_query(
    "SELECT * FROM proadel.DESTINOS_CLIENTES",
    "Destinations Data"
)

print(f"\nShapes loaded:")
print(f"  Customers    : {df_customers.shape}")
print(f"  Payments     : {df_payments.shape}")
print(f"  Credits      : {df_credits.shape}")
print(f"  Destinations : {df_destinations.shape}")


--- Success: [Customer Catalog] retrieved with 404 rows. ---
--- Success: [Payment (Cobranza) Data] retrieved with 12022 rows. ---
--- Success: [Credit (Ventas) Data] retrieved with 16476 rows. ---
--- Success: [Destinations Data] retrieved with 35 rows. ---

Shapes loaded:
  Customers    : (404, 11)
  Payments     : (12022, 14)
  Credits      : (16476, 18)
  Destinations : (35, 2)


In [ ]:

# =============================================================================
# SECTION 2: DATA CLEANING & TYPE NORMALIZATION
# =============================================================================

# -- 2.1  Sanitize column names that contain spaces -----------------------
df_payments = df_payments.rename(columns={
    "FORMA DE PAGO"    : "FORMA_DE_PAGO",
    "CONCEPTO DIVERSO" : "CONCEPTO_DIVERSO",
})

# -- 2.2  Parse FECHA columns (stored as nvarchar in the source system) ---
df_payments["FECHA"] = pd.to_datetime(df_payments["FECHA"], dayfirst=True, errors="coerce")
df_credits["FECHA"]  = pd.to_datetime(df_credits["FECHA"],  dayfirst=True, errors="coerce")

# -- 2.3  Numeric coercion for catalog amount columns ---------------------
money_cols = [
    "LIMITE_DE_CREDITO", "TOTAL_CREDITO", "TOTAL_COBRANZA",
    "TOTAL_DEV_VENTA", "SALDO_INICIAL", "TOTAL_POR_ABONAR",
]
for col in money_cols:
    df_customers[col] = pd.to_numeric(df_customers[col], errors="coerce").fillna(0)

df_payments["IMPORTE"] = pd.to_numeric(df_payments["IMPORTE"], errors="coerce").fillna(0)
df_credits["TOTAL"]    = pd.to_numeric(df_credits["TOTAL"],    errors="coerce").fillna(0)
df_credits["IMPORTE"]  = pd.to_numeric(df_credits["IMPORTE"],  errors="coerce").fillna(0)

# -- 2.4  Drop cancelled records where applicable -------------------------
if "ESTADO" in df_payments.columns:
    df_payments = df_payments[
        df_payments["ESTADO"].str.strip().str.upper() != "CANCELADO"
    ].copy()
if "ESTADO" in df_credits.columns:
    df_credits = df_credits[
        df_credits["ESTADO"].str.strip().str.upper() != "CANCELADO"
    ].copy()

print(f"Clean records — Payments: {len(df_payments):,}  |  Credits: {len(df_credits):,}")
print(f"Date range — Payments : {df_payments['FECHA'].min().date()} → {df_payments['FECHA'].max().date()}")
print(f"Date range — Credits  : {df_credits['FECHA'].min().date()}  → {df_credits['FECHA'].max().date()}")


In [ ]:

# =============================================================================
# SECTION 3: CUSTOMER-LEVEL FEATURE ENGINEERING
# Build the master 'features' DataFrame used by all downstream cells.
# =============================================================================

# ---------------------------------------------------------------------------
# 3.1  Payment aggregates per customer (from COBRANZA_DATA)
# ---------------------------------------------------------------------------
pay_agg = (
    df_payments
    .groupby("CLIENTE", as_index=False)
    .agg(
        payment_count      = ("IMPORTE", "count"),
        total_payments     = ("IMPORTE", "sum"),
        avg_payment        = ("IMPORTE", "mean"),
        last_payment_date  = ("FECHA",   "max"),
        first_payment_date = ("FECHA",   "min"),
    )
)
pay_agg["days_since_last_payment"] = (
    ANALYSIS_DATE - pay_agg["last_payment_date"]
).dt.days.clip(lower=0)
pay_agg["payment_span_days"] = (
    (pay_agg["last_payment_date"] - pay_agg["first_payment_date"])
    .dt.days.clip(lower=1)
)
pay_agg["avg_days_between_payments"] = (
    pay_agg["payment_span_days"] / pay_agg["payment_count"].clip(lower=1)
)

# ---------------------------------------------------------------------------
# 3.2  Credit/sales aggregates per customer (from CREDITO_DATA)
# ---------------------------------------------------------------------------
cred_agg = (
    df_credits
    .groupby("CLIENTE", as_index=False)
    .agg(
        credit_count        = ("TOTAL",  "count"),
        total_purchases     = ("TOTAL",  "sum"),
        avg_invoice         = ("TOTAL",  "mean"),
        last_purchase_date  = ("FECHA",  "max"),
        first_purchase_date = ("FECHA",  "min"),
    )
)
cred_agg["recency_days"] = (
    ANALYSIS_DATE - cred_agg["last_purchase_date"]
).dt.days.clip(lower=0)
# Tenure in months between first and last purchase
cred_agg["tenure_months"] = (
    (cred_agg["last_purchase_date"] - cred_agg["first_purchase_date"])
    .dt.days / 30.44
).clip(lower=1)
cred_agg["monthly_purchase_rate"] = (
    cred_agg["total_purchases"] / cred_agg["tenure_months"]
)

# ---------------------------------------------------------------------------
# 3.3  Merge into master feature table (customer catalog is the spine)
# ---------------------------------------------------------------------------
features = (
    df_customers[[
        "CODIGO", "NOMBRE", "ESTADO", "LIMITE_DE_CREDITO",
        "TOTAL_CREDITO", "TOTAL_COBRANZA", "SALDO_INICIAL", "TOTAL_POR_ABONAR"
    ]]
    .merge(pay_agg.rename(columns={"CLIENTE": "CODIGO"}),  on="CODIGO", how="left")
    .merge(cred_agg.rename(columns={"CLIENTE": "CODIGO"}), on="CODIGO", how="left")
)

# Fill zeros for customers with no transaction history
zero_cols = [
    "payment_count", "total_payments", "avg_payment",
    "credit_count", "total_purchases", "avg_invoice", "monthly_purchase_rate",
]
features[zero_cols] = features[zero_cols].fillna(0)

# For 'days' columns: dormant customers get worst-case value (999)
features["days_since_last_payment"]   = features["days_since_last_payment"].fillna(999)
features["avg_days_between_payments"] = features["avg_days_between_payments"].fillna(999)
features["recency_days"]              = features["recency_days"].fillna(999)

# ---------------------------------------------------------------------------
# 3.4  Derived risk metrics
# ---------------------------------------------------------------------------

# Payment/Purchase Ratio (PPR): fraction of invoiced amount that has been paid
features["payment_purchase_ratio"] = np.where(
    features["total_purchases"] > 0,
    (features["total_payments"] / features["total_purchases"]).clip(upper=1.50),
    0.0
)

# Net balance: total still owed (from the catalog field)
features["net_balance"] = features["TOTAL_POR_ABONAR"].clip(lower=0)

# Credit utilization: net balance vs. assigned credit limit
features["credit_utilization"] = np.where(
    features["LIMITE_DE_CREDITO"] > 0,
    (features["net_balance"] / features["LIMITE_DE_CREDITO"]).clip(upper=3.0),
    0.0
)

# Weighted Payment Frequency (WPF):
#   WPF = (payment_count * avg_payment) / (outstanding_balance + 1)
#   Higher value → more consistent, proportional repayments
features["weighted_payment_freq"] = (
    (features["payment_count"] * features["avg_payment"])
    / (features["net_balance"] + 1)
)

print(f"Master feature table: {features.shape[0]} customers × {features.shape[1]} columns")
features[[
    "CODIGO", "NOMBRE", "payment_purchase_ratio",
    "net_balance", "credit_utilization", "weighted_payment_freq", "recency_days"
]].head(6)


In [ ]:

# =============================================================================
# SECTION 3 (cont.): K-MEANS CLUSTERING
# =============================================================================

# Behavioural features used for segmentation
CLUSTER_FEATURES = [
    "payment_purchase_ratio",
    "credit_utilization",
    "weighted_payment_freq",
    "recency_days",
    "avg_days_between_payments",
    "net_balance",
]

# Replace infinities; log-scale monetary features to reduce skew
cluster_df = features[CLUSTER_FEATURES].copy()
cluster_df["net_balance"] = np.log1p(cluster_df["net_balance"])
cluster_df = cluster_df.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)

# --- Elbow + Silhouette to find optimal k --------------------------------
inertias, sil_scores = [], []
K_RANGE = range(2, 9)

for k in K_RANGE:
    km     = KMeans(n_clusters=k, n_init=15, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(K_RANGE), inertias, marker="o", color="steelblue", linewidth=2)
ax1.set_title("Elbow Method — Within-Cluster Inertia")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia")
ax1.grid(True, linestyle="--", alpha=0.4)

best_k_idx = sil_scores.index(max(sil_scores))
ax2.plot(list(K_RANGE), sil_scores, marker="s", color="mediumseagreen", linewidth=2)
ax2.set_title("Silhouette Score by k")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score (higher = better)")
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.axvline(list(K_RANGE)[best_k_idx], color="red", linestyle="--",
            linewidth=1.5, label=f"Best k = {list(K_RANGE)[best_k_idx]}")
ax2.legend()

plt.suptitle("K-Means: Optimal Cluster Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# --- Fit final model with best k -----------------------------------------
OPTIMAL_K = list(K_RANGE)[best_k_idx]
print(f"\nBest k by Silhouette Score: {OPTIMAL_K}  (score = {max(sil_scores):.3f})")

km_final = KMeans(n_clusters=OPTIMAL_K, n_init=20, random_state=42)
features["cluster"] = km_final.fit_predict(X_scaled)

print(f"\nCluster distribution:\n{features['cluster'].value_counts().sort_index().to_string()}")


## Section 1 — Exploratory Data Analysis & Data Cleaning

Before building any model we audit each table for type correctness, missing values, date ranges, and cancelled records.


In [ ]:

# =============================================================================
# SECTION 1: Data Cleaning
# =============================================================================

# --- 1.1  Parse dates (stored as nvarchar in the source DB) ---
for df in [df_collections, df_sales]:
    df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce', dayfirst=False)

# --- 1.2  Rename columns with spaces for easier access ---
df_collections = df_collections.rename(columns={
    'FORMA DE PAGO'  : 'FORMA_PAGO',
    'CONCEPTO DIVERSO': 'CONCEPTO_DIVERSO'
})

# --- 1.3  Coerce numeric types ---
numeric_catalog = ['LIMITE_DE_CREDITO', 'TOTAL_CREDITO', 'TOTAL_COBRANZA',
                   'TOTAL_DEV_VENTA', 'SALDO_INICIAL', 'TOTAL_POR_ABONAR']
for col in numeric_catalog:
    df_clients[col] = pd.to_numeric(df_clients[col], errors='coerce').fillna(0)

df_collections['IMPORTE'] = pd.to_numeric(df_collections['IMPORTE'], errors='coerce').fillna(0)
for col in ['IMPORTE', 'TOTAL', 'PRECIO', 'CANTIDAD']:
    df_sales[col] = pd.to_numeric(df_sales[col], errors='coerce').fillna(0)

# --- 1.4  Remove CANCELADO records from both transaction tables ---
df_sales_clean       = df_sales[df_sales['ESTADO'].str.upper()            != 'CANCELADO'].copy()
df_collections_clean = df_collections[df_collections['ESTADO'].str.upper() != 'CANCELADO'].copy()

# --- 1.5  Active clients only ---
df_clients_active = df_clients[df_clients['ESTADO'].str.upper() == 'ACTIVO'].copy()

print(f"Active clients        : {len(df_clients_active):>5,}")
print(f"Clean sales rows      : {len(df_sales_clean):>5,}")
print(f"Clean collection rows : {len(df_collections_clean):>5,}")

# --- 1.6  Missing-value heatmap ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (title, df) in zip(axes, [
        ("Clients (numeric)", df_clients_active[numeric_catalog]),
        ("Collections",       df_collections_clean[['IMPORTE', 'FECHA']]),
        ("Sales",             df_sales_clean[['IMPORTE', 'TOTAL', 'PRECIO', 'FECHA']])]):
    sns.heatmap(df.isnull(), cbar=False, ax=ax, yticklabels=False, cmap='Reds')
    ax.set_title(f'Missing values — {title}')
plt.tight_layout()
plt.show()

# --- 1.7  Date range ---
print(f"\nSales date range       : {df_sales_clean['FECHA'].min().date()} → {df_sales_clean['FECHA'].max().date()}")
print(f"Collections date range : {df_collections_clean['FECHA'].min().date()} → {df_collections_clean['FECHA'].max().date()}")


## Section 2 — Feature Engineering

Building a customer-level behavioral feature matrix from transaction history.

| Feature | Description |
|---|---|
| `total_purchases` | Total billed sales amount |
| `total_payments` | Total payments/collections received |
| `net_balance` | Estimated outstanding balance (purchases − payments) |
| `payment_purchase_ratio` | % of debt covered by payments (0–1.5) |
| `n_purchase_txns` | Number of distinct purchase invoices |
| `n_payment_txns` | Number of payment transactions |
| `avg_days_between_payments` | Mean inter-payment interval in days |
| `weighted_payment_freq` | Cadence index: (n_payments × avg_amount) / net_balance |
| `recency_days` | Days since last purchase |
| `credit_utilization` | Net balance / assigned credit limit |
| `monthly_purchase_rate` | Average monthly purchase volume |


In [ ]:

# =============================================================================
# SECTION 2: Feature Engineering
# =============================================================================

# Reference date: latest transaction across both tables
ANALYSIS_DATE = max(
    df_sales_clean['FECHA'].max(),
    df_collections_clean['FECHA'].max()
)
print(f"Analysis reference date: {ANALYSIS_DATE.date()}")

# ---------------------------------------------------------------------------
# 2.1  Purchase aggregates per customer
# ---------------------------------------------------------------------------
sales_agg = (
    df_sales_clean
    .groupby('CLIENTE')
    .agg(
        total_purchases     = ('IMPORTE', 'sum'),
        n_purchase_txns     = ('FOLIO',   'nunique'),
        last_purchase_date  = ('FECHA',   'max'),
        first_purchase_date = ('FECHA',   'min'),
        avg_ticket          = ('IMPORTE', 'mean')
    )
    .reset_index()
    .rename(columns={'CLIENTE': 'CODIGO'})
)

sales_agg['recency_days']  = (ANALYSIS_DATE - sales_agg['last_purchase_date']).dt.days
sales_agg['active_months'] = (
    (sales_agg['last_purchase_date'] - sales_agg['first_purchase_date']).dt.days / 30.44
).clip(lower=1)

# ---------------------------------------------------------------------------
# 2.2  Payment aggregates per customer
# ---------------------------------------------------------------------------
pay_agg = (
    df_collections_clean
    .groupby('CLIENTE')
    .agg(
        total_payments    = ('IMPORTE', 'sum'),
        n_payment_txns    = ('ID',      'count'),
        last_payment_date = ('FECHA',   'max')
    )
    .reset_index()
    .rename(columns={'CLIENTE': 'CODIGO'})
)

# ---------------------------------------------------------------------------
# 2.3  Average days between consecutive payments per customer
# ---------------------------------------------------------------------------
pay_sorted = (
    df_collections_clean
    .sort_values(['CLIENTE', 'FECHA'])
    .groupby('CLIENTE')['FECHA']
    .apply(lambda x: x.diff().dt.days.dropna().mean())
    .reset_index()
    .rename(columns={'CLIENTE': 'CODIGO', 'FECHA': 'avg_days_between_payments'})
)

# ---------------------------------------------------------------------------
# 2.4  Merge everything with the clients catalog
# ---------------------------------------------------------------------------
features = (
    df_clients_active[['CODIGO', 'NOMBRE', 'LIMITE_DE_CREDITO',
                        'TOTAL_CREDITO', 'TOTAL_COBRANZA', 'TOTAL_DEV_VENTA',
                        'SALDO_INICIAL', 'TOTAL_POR_ABONAR']]
    .merge(sales_agg,  on='CODIGO', how='left')
    .merge(pay_agg,    on='CODIGO', how='left')
    .merge(pay_sorted, on='CODIGO', how='left')
)

# Fill NaN for customers with no recorded transactions
num_fill = ['total_purchases', 'n_purchase_txns', 'total_payments',
            'n_payment_txns', 'avg_days_between_payments', 'avg_ticket']
features[num_fill]        = features[num_fill].fillna(0)
features['recency_days']  = features['recency_days'].fillna(999)
features['active_months'] = features['active_months'].fillna(1)

# ---------------------------------------------------------------------------
# 2.5  Derived ratios and composite indices
# ---------------------------------------------------------------------------

# Payment-to-purchase ratio (capped at 1.5 to handle overpayments/refunds)
features['payment_purchase_ratio'] = np.where(
    features['total_purchases'] > 0,
    (features['total_payments'] / features['total_purchases']).clip(0, 1.5),
    0
)

# Net balance: total purchased minus total paid, floored at 0
features['net_balance'] = (features['total_purchases'] - features['total_payments']).clip(lower=0)

# Credit utilization: net balance relative to assigned limit
features['credit_utilization'] = np.where(
    features['LIMITE_DE_CREDITO'] > 0,
    features['net_balance'] / features['LIMITE_DE_CREDITO'],
    0
)

# Average amount per payment transaction
features['avg_payment_amount'] = np.where(
    features['n_payment_txns'] > 0,
    features['total_payments'] / features['n_payment_txns'],
    0
)

# Weighted payment frequency index: (n_payments × avg_amount) / net_balance
features['weighted_payment_freq'] = np.where(
    features['net_balance'] > 0,
    (features['n_payment_txns'] * features['avg_payment_amount']) / features['net_balance'],
    features['n_payment_txns'] * 0.1   # small baseline for zero-balance customers
)

# Monthly purchase rate: total purchases / active months
features['monthly_purchase_rate'] = features['total_purchases'] / features['active_months']

print(f"\nFeature matrix shape: {features.shape}")
key_feats = ['total_purchases', 'total_payments', 'payment_purchase_ratio',
             'net_balance', 'credit_utilization', 'weighted_payment_freq',
             'recency_days', 'avg_days_between_payments']
features[key_feats].describe().round(2)


In [ ]:

# =============================================================================
# SECTION 2 (cont.): Feature Distributions
# =============================================================================

plot_features = [
    ('total_purchases',           'Total Purchases',             True),
    ('total_payments',            'Total Payments',              True),
    ('payment_purchase_ratio',    'Payment/Purchase Ratio',      False),
    ('net_balance',               'Net Balance',                 True),
    ('credit_utilization',        'Credit Utilization',          False),
    ('weighted_payment_freq',     'Weighted Payment Freq Index', False),
    ('recency_days',              'Recency (days)',               False),
    ('avg_days_between_payments', 'Avg Days Between Payments',   False),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for ax, (col, label, log_scale) in zip(axes, plot_features):
    data = features[col].replace([np.inf, -np.inf], np.nan).dropna()
    if log_scale:
        data  = np.log1p(data)
        label = f'log(1+{label})'
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Customer Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Section 3 — Customer Segmentation via K-Means

We cluster customers using standardized behavioral features.  
The **Elbow Method** and **Silhouette Score** jointly determine the optimal number of segments *k*.


In [ ]:

# =============================================================================
# SECTION 3: K-Means Clustering
# =============================================================================

# --- 3.1  Select and prepare clustering features ---
CLUSTER_FEATURES = [
    'total_purchases',            # purchase volume
    'total_payments',             # payment volume
    'payment_purchase_ratio',     # repayment discipline
    'net_balance',                # current credit exposure
    'n_purchase_txns',            # purchase frequency
    'avg_days_between_payments',  # payment cadence
    'recency_days',               # activity recency
    'credit_utilization',         # assigned limit utilization
]

# Clip extreme outliers to 99th percentile
cluster_df = features[CLUSTER_FEATURES].copy()
for col in CLUSTER_FEATURES:
    p99 = cluster_df[col].quantile(0.99)
    cluster_df[col] = cluster_df[col].clip(upper=p99).replace([np.inf, -np.inf], 0).fillna(0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)

# --- 3.2  Elbow Method + Silhouette Score sweep ---
inertias, silhouettes = [], []
K_range  = range(2, 10)
n_sample = min(500, len(X_scaled))

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl, sample_size=n_sample, random_state=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.4)

ax2.plot(list(K_range), silhouettes, 's-', color='darkorange', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score by k', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

# --- 3.3  Fit optimal k (max silhouette) ---
OPTIMAL_K = list(K_range)[silhouettes.index(max(silhouettes))]
print(f"Optimal k  (max silhouette = {max(silhouettes):.3f}): {OPTIMAL_K}")

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
features['cluster'] = kmeans.fit_predict(X_scaled)

print("\nCluster size distribution:")
print(features['cluster'].value_counts().sort_index())


In [ ]:

# =============================================================================
# SECTION 3 (cont.): Cluster Visualization — Scatter Plots
# =============================================================================

pca      = PCA(n_components=2, random_state=42)
X_pca    = pca.fit_transform(X_scaled)
features['pca_1'] = X_pca[:, 0]
features['pca_2'] = X_pca[:, 1]

palette = sns.color_palette("Set2", OPTIMAL_K)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA 2D projection
for c in range(OPTIMAL_K):
    mask = features['cluster'] == c
    axes[0].scatter(
        features.loc[mask, 'pca_1'], features.loc[mask, 'pca_2'],
        label=f'Cluster {c}', alpha=0.7, s=60, color=palette[c]
    )
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
axes[0].set_title('Customer Clusters — PCA Projection', fontsize=13, fontweight='bold')
axes[0].legend(title='Cluster')
axes[0].grid(True, alpha=0.3)

# Purchase volume vs. repayment behavior
sc = axes[1].scatter(
    np.log1p(features['total_purchases']),
    features['payment_purchase_ratio'],
    c=features['cluster'], cmap='Set2', alpha=0.7, s=60
)
axes[1].set_xlabel('log(1 + Total Purchases)', fontsize=11)
axes[1].set_ylabel('Payment / Purchase Ratio', fontsize=11)
axes[1].set_title('Volume vs. Repayment Behavior by Cluster', fontsize=13, fontweight='bold')
axes[1].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Good payer threshold (0.80)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()


In [ ]:

# =============================================================================
# SECTION 3 (cont.): Cluster Descriptive Statistics & Profile Heatmap
# =============================================================================

cluster_means = features.groupby('cluster')[CLUSTER_FEATURES].mean().round(2)

print("=== Cluster Means ===")
print(cluster_means.T.to_string())

# Normalize 0–1 for visual coloring
cluster_norm = (cluster_means - cluster_means.min()) / (
    cluster_means.max() - cluster_means.min() + 1e-9
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    cluster_norm.T,
    annot=cluster_means.T.round(1),
    fmt='g',
    cmap='RdYlGn',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Normalized Score (0 = low, 1 = high)'}
)
ax.set_title('Cluster Profiles — Normalized Feature Heatmap', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.show()

# --- Auto-derive descriptive labels from cluster statistics ---
cluster_labels = {}
for c in range(OPTIMAL_K):
    row    = cluster_means.loc[c]
    ppr_med = cluster_means['payment_purchase_ratio'].median()
    vol_med = cluster_means['total_purchases'].median()
    rec_med = cluster_means['recency_days'].median()
    bal_med = cluster_means['net_balance'].median()
    if row['payment_purchase_ratio'] >= ppr_med and row['net_balance'] <= bal_med:
        cluster_labels[c] = 'High-value / Good payer'
    elif row['total_purchases'] >= vol_med and row['payment_purchase_ratio'] < ppr_med:
        cluster_labels[c] = 'High-volume / Slow payer'
    elif row['total_purchases'] < vol_med and row['recency_days'] > rec_med:
        cluster_labels[c] = 'Dormant / Low engagement'
    else:
        cluster_labels[c] = 'Moderate risk / Regular buyer'

features['cluster_label'] = features['cluster'].map(cluster_labels)

print("\nDerived cluster labels:")
for k, v in cluster_labels.items():
    n = (features['cluster'] == k).sum()
    print(f"  Cluster {k}: {v:<35s}  ({n} customers)")


## Section 4 — Payment & Profitability Metrics

| Metric | Definition |
|---|---|
| **Payment/Purchase Ratio (PPR)** | Total payments ÷ Total purchases — ideal > 0.80 |
| **Weighted Payment Freq Index** | Captures both cadence and magnitude of payments relative to outstanding balance |
| **Cash Flow Score** | Normalized total payments — measures immediate liquidity contribution |
| **Profitability Score** | Monthly purchase rate × payment reliability — long-term value proxy |


In [ ]:

# =============================================================================
# SECTION 4: Payment & Profitability Metrics
# =============================================================================

palette = sns.color_palette("Set2", OPTIMAL_K)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 4.1  Payment/Purchase Ratio distribution per cluster ---
for c in range(OPTIMAL_K):
    mask = features['cluster'] == c
    axes[0].hist(
        features.loc[mask, 'payment_purchase_ratio'],
        bins=20, alpha=0.65,
        label=f'C{c}: {cluster_labels[c][:22]}'
    )
axes[0].axvline(0.8, color='red', linestyle='--', linewidth=1.5, label='Good threshold (0.80)')
axes[0].set_title('Payment/Purchase Ratio by Cluster', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Ratio')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# --- 4.2  Weighted payment frequency index vs. PPR ---
axes[1].scatter(
    features['weighted_payment_freq'].clip(upper=features['weighted_payment_freq'].quantile(0.95)),
    features['payment_purchase_ratio'],
    c=features['cluster'], cmap='Set2', alpha=0.65, s=50
)
axes[1].set_xlabel('Weighted Payment Freq Index', fontsize=11)
axes[1].set_ylabel('Payment / Purchase Ratio', fontsize=11)
axes[1].set_title('Payment Cadence vs. Coverage', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

# --- 4.3  Net balance box plot per cluster ---
log_balance = np.log1p(features['net_balance'])
sns.boxplot(x=features['cluster'], y=log_balance, palette='Set2', ax=axes[2])
axes[2].set_xlabel('Cluster', fontsize=11)
axes[2].set_ylabel('log(1 + Net Balance)', fontsize=11)
axes[2].set_title('Outstanding Balance by Cluster', fontsize=12, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# 4.4  Cash Flow Score vs. Long-term Profitability Score
# ---------------------------------------------------------------------------
# Immediate cash flow: how much cash this customer has contributed relative to the portfolio
features['cash_flow_score'] = (
    features['total_payments'] / (features['total_payments'].max() + 1)
) * 100

# Long-term profitability: monthly revenue × payment reliability
prof_raw = features['monthly_purchase_rate'] * features['payment_purchase_ratio'].clip(0, 1)
max_prof  = prof_raw.quantile(0.99)
features['profitability_score'] = (prof_raw / (max_prof + 1)).clip(0, 1) * 100

top_cf   = features.nlargest(15, 'cash_flow_score')
top_prof = features.nlargest(15, 'profitability_score')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top_cf['NOMBRE'].str[:30],   top_cf['cash_flow_score'],
             color=[palette[c] for c in top_cf['cluster']], alpha=0.85)
axes[0].set_title('Top 15 — Immediate Cash Flow Score',      fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score (0–100)')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(top_prof['NOMBRE'].str[:30], top_prof['profitability_score'],
             color=[palette[c] for c in top_prof['cluster']], alpha=0.85)
axes[1].set_title('Top 15 — Long-term Profitability Score',  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Score (0–100)')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# --- 4.5  Cash Flow vs. Profitability quadrant matrix ---
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    features['cash_flow_score'], features['profitability_score'],
    c=features['cluster'], cmap='Set2', alpha=0.7, s=70
)
cf_med = features['cash_flow_score'].median()
pf_med = features['profitability_score'].median()
ax.axvline(cf_med, color='gray', linestyle='--', alpha=0.5)
ax.axhline(pf_med, color='gray', linestyle='--', alpha=0.5)

ax.text(cf_med * 0.05, pf_med * 1.05, 'High Profit\nLow Cash',    fontsize=9, color='forestgreen')
ax.text(cf_med * 1.05, pf_med * 1.05, 'STARS\n(Profit + Cash)',   fontsize=9, color='darkgreen', fontweight='bold')
ax.text(cf_med * 0.05, pf_med * 0.15, 'Low Value',                fontsize=9, color='red')
ax.text(cf_med * 1.05, pf_med * 0.15, 'Cash Cows\n(High Cash)',   fontsize=9, color='darkorange')

ax.set_xlabel('Cash Flow Score (immediate liquidity)',  fontsize=12)
ax.set_ylabel('Long-term Profitability Score',          fontsize=12)
ax.set_title('Cash Flow vs. Profitability Matrix',      fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Section 5 — Credit Risk Model & Optimal Limit Policy

### Methodology

1. **Breakpoint Detection** — Customers are binned by net balance; we measure where the average Payment/Purchase Ratio begins to degrade statistically.
2. **Composite Risk Score** — Weighted sum of four normalized sub-scores:

| Sub-score | Weight | Measures |
|---|---|---|
| `sc_payment` | 40 % | Payment deficit (1 − PPR) |
| `sc_utilization` | 25 % | Credit utilization ratio |
| `sc_recency` | 20 % | Days since last purchase |
| `sc_cadence` | 15 % | Average days between payments |

3. **Traffic-light Tiers**

| Tier | Risk Score | Credit Action |
|---|---|---|
| **Green** | < 30 | Maintain or expand credit line |
| **Yellow** | 30–60 | Restrict credit; manual review required |
| **Red** | > 60 | Cash-only sales; stop credit immediately |

### Optimal Credit Limit Formula

```
optimal_limit = monthly_purchase_rate × payment_purchase_ratio × coverage_factor
```

Coverage factors → **Green: 3×**, **Yellow: 1.5×**, **Red: 0×**


In [ ]:

# =============================================================================
# SECTION 5: Composite Risk Scoring & Breakpoint Detection
# =============================================================================

# ---------------------------------------------------------------------------
# 5.1  Sub-scores — each normalized 0–1 (1 = highest risk)
# ---------------------------------------------------------------------------

# a) Payment deficit: how much of debt is NOT covered
features['sc_payment'] = (1 - features['payment_purchase_ratio'].clip(0, 1))

# b) Credit utilization: net balance relative to assigned limit
features['sc_utilization'] = features['credit_utilization'].clip(0, 2) / 2

# c) Recency: more days since last purchase → higher risk
max_recency = features['recency_days'].quantile(0.95)
features['sc_recency'] = features['recency_days'].clip(0, max_recency) / max_recency

# d) Payment cadence: longer gaps between payments → higher risk
max_cadence = features['avg_days_between_payments'].replace(0, np.nan).quantile(0.95)
features['sc_cadence'] = (
    features['avg_days_between_payments']
    .clip(0, max_cadence).fillna(max_cadence) / (max_cadence + 1)
)

# Weighted composite risk score (0–100)
WEIGHTS = {'sc_payment': 0.40, 'sc_utilization': 0.25, 'sc_recency': 0.20, 'sc_cadence': 0.15}
features['risk_score'] = (
    sum(features[col] * w for col, w in WEIGHTS.items()) * 100
).round(2)

# ---------------------------------------------------------------------------
# 5.2  Breakpoint detection: balance quantile vs. avg payment ratio
# ---------------------------------------------------------------------------
# Bin customers by net balance and measure where the average PPR starts degrading
balance_bins = pd.qcut(features['net_balance'].clip(lower=0.01), q=10, duplicates='drop')
bp_df = (
    features.assign(balance_bin=balance_bins)
    .groupby('balance_bin', observed=True)
    .agg(avg_ppr    = ('payment_purchase_ratio', 'mean'),
         avg_balance= ('net_balance',            'mean'),
         n_customers= ('CODIGO',                 'count'))
    .reset_index()
)
bp_df['ppr_delta'] = bp_df['avg_ppr'].diff()
worst_idx          = bp_df['ppr_delta'].idxmin()
breakpoint_balance = bp_df.loc[worst_idx, 'avg_balance']
breakpoint_drop    = abs(bp_df.loc[worst_idx, 'ppr_delta'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Breakpoint bar chart: payment ratio per balance quantile
bar_colors = ['#2ecc71' if r >= 0.75 else '#e67e22' if r >= 0.50 else '#e74c3c'
              for r in bp_df['avg_ppr']]
axes[0].bar(range(len(bp_df)), bp_df['avg_ppr'], color=bar_colors, edgecolor='white', alpha=0.9)
axes[0].axvline(worst_idx - 0.5, color='black', linestyle='--', linewidth=2,
                label=f'Breakpoint ≈ ${breakpoint_balance:,.0f}')
axes[0].set_xticks(range(len(bp_df)))
axes[0].set_xticklabels([f'Q{i+1}' for i in range(len(bp_df))], rotation=45, ha='right')
axes[0].set_ylabel('Avg Payment / Purchase Ratio')
axes[0].set_title('Payment Degradation by Balance Quantile\n(statistical breakpoint)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Risk score distribution with tier thresholds
axes[1].hist(features['risk_score'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(30, color='#2ecc71', linestyle='--', linewidth=2, label='Green threshold (30)')
axes[1].axvline(60, color='#e67e22', linestyle='--', linewidth=2, label='Yellow threshold (60)')
axes[1].set_title('Composite Risk Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Risk Score  (0 = safest, 100 = riskiest)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Breakpoint balance  : ${breakpoint_balance:,.2f}")
print(f"PPR drop at breakpt : {breakpoint_drop:.1%}  (above this threshold, repayment behavior degrades)")


In [ ]:

# =============================================================================
# SECTION 5 (cont.): Credit Classification & Optimal Limit Recommendation
# =============================================================================

# --- 5.3  Traffic-light classification thresholds ---
def classify_risk(score):
    if   score < 30: return 'Green'
    elif score < 60: return 'Yellow'
    else:            return 'Red'

features['risk_tier'] = features['risk_score'].apply(classify_risk)

# Coverage multipliers per tier (drives the optimal limit formula)
COVERAGE = {'Green': 3.0, 'Yellow': 1.5, 'Red': 0.0}

# --- 5.4  Optimal credit limit formula ---
# optimal_limit = monthly_purchase_rate * payment_purchase_ratio * coverage_factor
features['optimal_credit_limit'] = (
    features['monthly_purchase_rate']
    * features['payment_purchase_ratio'].clip(0.10, 1.0)  # floor at 10% for new clients
    * features['risk_tier'].map(COVERAGE)
).round(2)

# Green customers: never recommend below 80 % of their current assigned limit
green_mask = features['risk_tier'] == 'Green'
features.loc[green_mask, 'optimal_credit_limit'] = features.loc[green_mask].apply(
    lambda r: max(r['optimal_credit_limit'], r['LIMITE_DE_CREDITO'] * 0.8), axis=1
)
# Red customers: hard stop at zero
features.loc[features['risk_tier'] == 'Red', 'optimal_credit_limit'] = 0.0

# --- 5.5  Risk factor correlation heatmap ---
risk_cols = [
    'payment_purchase_ratio', 'net_balance', 'credit_utilization',
    'weighted_payment_freq', 'recency_days', 'avg_days_between_payments', 'risk_score'
]
corr_matrix = features[risk_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5,
    mask=np.triu(np.ones_like(corr_matrix, dtype=bool), k=1),
    ax=ax, vmin=-1, vmax=1
)
ax.set_title('Risk Factor Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- 5.6  Tier breakdown summary & charts ---
tier_summary = features.groupby('risk_tier').agg(
    n_customers       = ('CODIGO',                'count'),
    avg_ppr           = ('payment_purchase_ratio', 'mean'),
    avg_balance       = ('net_balance',            'mean'),
    avg_current_limit = ('LIMITE_DE_CREDITO',      'mean'),
    avg_optimal_limit = ('optimal_credit_limit',   'mean'),
    total_exposure    = ('net_balance',            'sum')
).round(2)

print("\n=== Risk Tier Summary ===")
print(tier_summary.to_string())

tier_colors_map = {'Green': '#2ecc71', 'Yellow': '#f1c40f', 'Red': '#e74c3c'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pie: customer count distribution
tier_counts = features['risk_tier'].value_counts()
axes[0].pie(
    tier_counts.values,
    labels=tier_counts.index,
    colors=[tier_colors_map[t] for t in tier_counts.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Customer Distribution by Risk Tier', fontsize=12, fontweight='bold')

# Bar: current vs. recommended limit per tier
x = np.arange(len(tier_summary))
w = 0.35
axes[1].bar(x - w/2, tier_summary['avg_current_limit'], w,
            label='Current Limit',      color='steelblue',  alpha=0.85)
axes[1].bar(x + w/2, tier_summary['avg_optimal_limit'], w,
            label='Recommended Limit',  color='darkorange',  alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(tier_summary.index)
axes[1].set_title('Avg Credit Limit: Current vs. Recommended', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amount ($)')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[1].grid(axis='y', alpha=0.3)

# Bar: total portfolio exposure per tier
axes[2].bar(
    tier_summary.index, tier_summary['total_exposure'],
    color=[tier_colors_map.get(t, 'gray') for t in tier_summary.index],
    edgecolor='white', linewidth=1.5, alpha=0.9
)
axes[2].set_title('Total Portfolio Exposure by Tier', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Total Net Balance ($)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# =============================================================================
# SECTION 6: Final Summary Table — Credit Limit Recommendations
# =============================================================================

SUMMARY_COLS = [
    'CODIGO', 'NOMBRE',
    'total_purchases', 'total_payments', 'payment_purchase_ratio',
    'net_balance', 'credit_utilization',
    'weighted_payment_freq', 'recency_days',
    'LIMITE_DE_CREDITO', 'optimal_credit_limit',
    'cluster', 'cluster_label',
    'risk_score', 'risk_tier'
]

summary_table = features[SUMMARY_COLS].copy()
summary_table = summary_table.sort_values(['risk_tier', 'risk_score'])
summary_table = summary_table.rename(columns={
    'CODIGO'                : 'customer_id',
    'NOMBRE'                : 'name',
    'total_purchases'       : 'total_purchases_$',
    'total_payments'        : 'total_payments_$',
    'payment_purchase_ratio': 'ppr',
    'net_balance'           : 'net_balance_$',
    'credit_utilization'    : 'credit_util',
    'LIMITE_DE_CREDITO'     : 'current_limit_$',
    'optimal_credit_limit'  : 'recommended_limit_$',
})

# Apply display formatting on a copy for printing
display_df = summary_table.copy()
for col in ['total_purchases_$', 'total_payments_$', 'net_balance_$',
            'current_limit_$', 'recommended_limit_$']:
    display_df[col] = display_df[col].apply(lambda x: f'${x:,.2f}')
display_df['ppr']         = display_df['ppr'].apply(lambda x: f'{x:.1%}')
display_df['credit_util'] = display_df['credit_util'].apply(lambda x: f'{x:.1%}')
display_df['risk_score']  = display_df['risk_score'].apply(lambda x: f'{x:.1f}')

# --- Color-coded console report ---
print("=" * 95)
print(f"  CREDIT LIMIT RECOMMENDATION REPORT  |  Analysis Date: {ANALYSIS_DATE.date()}  |  Customers: {len(display_df)}")
print("=" * 95)

for tier, esc in [('Green', '\033[92m'), ('Yellow', '\033[93m'), ('Red', '\033[91m')]:
    sub = display_df[display_df['risk_tier'] == tier]
    print(f"\n{esc}{'─'*28} {tier.upper()} TIER  ({len(sub)} customers) {'─'*28}\033[0m")
    cols = ['customer_id', 'name', 'ppr', 'net_balance_$',
            'current_limit_$', 'recommended_limit_$', 'risk_score']
    print(sub[cols].to_string(index=False, max_colwidth=28))

# --- Export to CSV (raw numbers, no string formatting) ---
export_path = 'credit_limit_recommendations.csv'
summary_table.to_csv(export_path, index=False)
print(f"\n\nFull report exported to: {export_path}")
print(f"\nPortfolio snapshot:")
print(f"  Green  (expand / maintain) : {(features['risk_tier']=='Green').sum():>4} customers")
print(f"  Yellow (restrict / review) : {(features['risk_tier']=='Yellow').sum():>4} customers")
print(f"  Red    (cash only)         : {(features['risk_tier']=='Red').sum():>4} customers")
